# Building a cross-covariance

When two SOLikeT likelihoods analyse data drawn from the *same* sky, their measurement errors are
correlated. A joint analysis therefore needs the **cross-covariance** between their data vectors, not
just the two auto-covariances. This notebook builds the cross-covariance between the primary CMB
(MFLike TT/TE/EE) and the CMB-lensing reconstruction, which is dominated by the fact that lensing
smooths the primary acoustic peaks.

`soliket.cross_covariance` provides two layers:

- a **one-liner**, `CrossCov.from_cmb_lensing(session)`, that pulls everything it needs from a
  `Session` and returns a ready-to-save `CrossCov`;
- the **low-level kernels and builders** (`cmb_lensing_crosscov`, `lensing_induced_cov`,
  `shear_kappa_crosscov`, `camb_lensing_derivatives`, ...) for finer control.

The result is consumed by [`analyse_datasets.ipynb`](analyse_datasets.ipynb), which feeds it into a
`MultiGaussianLikelihood`.

## 1. The one-liner (full accuracy)

On real MFLike data the cross-covariance is driven by the CAMB *lensed-Cl derivative*
`d C_ell^XY / d C_L^phiphi`, evaluated at the MFLike `lmax` (~9000). That derivative is a dense
`(4, lmax+1, lmax+1)` array of roughly **3 GB** and takes several minutes. Since the three notebooks
are meant to run as one analysis, the cell below ships with `RUN_FULL = True` and writes the real
cross-covariance into `sims/` for the analysis to consume; set it to `False` to skip the heavy
derivative on a memory-constrained machine.

`CrossCov.from_cmb_lensing` takes the two evaluated likelihoods directly (`roles.mflike`,
`roles.lensing` from `resolve_aliases` — the concrete handles, not a `Session`).

In [ ]:
from pathlib import Path

RUN_FULL = True  # set False to skip the heavy (~3 GB, minutes) CAMB derivative

SIMS = Path("sims")
SIMS.mkdir(exist_ok=True)
XCOV = SIMS / "XCov_mflike_lensing.fits"

if RUN_FULL:
    from cobaya.model import get_model
    from cobaya.tools import resolve_packages_path

    from soliket.gaussian.gaussian_data import CrossCov
    from soliket.presets import build_info, resolve_aliases

    # ISO fiducial via the override folder; build the model and take the named
    # roles (the concrete handles, not a Session).
    info = build_info("multigaussian", defaults_dir="defaults")
    info["packages_path"] = resolve_packages_path()
    model = get_model(info)
    model.loglikes({})  # evaluate CMB + lensing at the fiducial point
    roles = resolve_aliases(model)

    # full-accuracy CAMB derivative (~3 GB, minutes)
    xcov = CrossCov.from_cmb_lensing(roles.mflike, roles.lensing)
    xcov.save(str(XCOV))
    cmb, lensing = xcov.component_names  # ("mflike", "CMB Lensing")
    print("saved cross-covariance", XCOV.name, "block:", xcov[(cmb, lensing)].shape)
else:
    print("RUN_FULL is False - skipping the full-accuracy CAMB derivative.")

## 2. Validating the saved product

A cheap sanity-check on the **actual** cross-covariance written above (no recomputation): load it back
and confirm it is self-consistent and usable in a joint fit — the cross block matches the two auto
blocks, the auto-covariances are symmetric, and the assembled joint covariance is finite and
invertible. We also report how strongly the two probes are correlated.

In [ ]:
from pathlib import Path

import numpy as np

from soliket.gaussian.gaussian_data import CrossCov

XCOV = Path("sims") / "XCov_mflike_lensing.fits"

xcov = CrossCov.load(str(XCOV))
cmb, lensing = xcov.component_names
A = np.asarray(xcov[(cmb, cmb)])  # MFLike auto-covariance
B = np.asarray(xcov[(lensing, lensing)])  # CMB-lensing auto-covariance
X = np.asarray(xcov[(cmb, lensing)])  # the CMB x lensing cross block

print("components  :", xcov.component_names)
print("auto blocks :", A.shape, "+", B.shape)
print("cross block :", X.shape, "(expect", (A.shape[0], B.shape[0]), ")")

# Structural checks: the saved product must be self-consistent and usable.
assert X.shape == (A.shape[0], B.shape[0]), "cross block does not match the auto blocks"
for name, M in [(cmb, A), (lensing, B)]:
    assert np.allclose(M, M.T), f"{name} auto-covariance is not symmetric"
full = np.block([[A, X], [X.T, B]])
assert np.all(np.isfinite(full)), "joint covariance has non-finite entries"
np.linalg.inv(full)  # raises LinAlgError if singular -> unusable in the fit

print(
    "joint cov   :",
    full.shape,
    "| min eig: {:.2e}".format(np.linalg.eigvalsh(full).min()),
    "| invertible: yes",
)

# How strongly are the two probes correlated? (dimensionless correlation block)
corr = X / np.sqrt(np.outer(np.diag(A), np.diag(B)))
print("max |corr(CMB, lensing)| : {:.3f}".format(np.abs(corr).max()))

### What the per-bandpower labels buy you

The saved product is more than a matrix: every block carries the **identity** of each bandpower it
spans. When `MultiGaussianLikelihood` assembles the joint covariance, `CrossCov.to_canonical` uses
these labels to realign each block to the data *by identity* — so the cross-covariance stays correct
even when a probe's data vector is in a different order than the block was built in (a reordered
`cov_Bbl`, TE/ET folding) or carries different scale cuts. The build order is no longer load-bearing;
the cross-cov can be produced in any order and is reshuffled to the canonical theory order on load.

In [ ]:
# The MFLike rows are labelled in mflike's own vocabulary (spectrum, channel pair,
# ell); the lensing columns in the SACC's (data_type, tracers, ell). These identities
# are what align each block to its likelihood's data vector -- by identity, not order.
cmb_ids = xcov.component_ids(cmb)
lens_ids = xcov.component_ids(lensing)

print(f"{cmb}: {len(cmb_ids)} labelled bandpowers, e.g.")
for key in cmb_ids[:3]:
    print("   ", key)
print(f"{lensing}: {len(lens_ids)} labelled bandpowers, e.g.")
for key in lens_ids[:3]:
    print("   ", key)

In [ ]:
import matplotlib.pyplot as plt

# Normalise the joint covariance to a correlation matrix (diag -> 1).
d = 1.0 / np.sqrt(np.diag(full))
corr_full = full * np.outer(d, d)
n_cmb = A.shape[0]
cmax = np.abs(corr).max()

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: the full joint correlation matrix, with the MFLike|lensing block divider.
im0 = ax0.imshow(corr_full, cmap="RdBu_r", vmin=-1, vmax=1)
ax0.axhline(n_cmb - 0.5, color="k", lw=0.8)
ax0.axvline(n_cmb - 0.5, color="k", lw=0.8)
ax0.set_title("Joint correlation matrix")
ax0.set_xlabel(f"{cmb}   |   {lensing}")
ax0.set_ylabel(f"{cmb}   |   {lensing}")
fig.colorbar(im0, ax=ax0, fraction=0.046, label="correlation")

# Right: the CMB x lensing cross-correlation block on its own (much smaller) scale.
im1 = ax1.imshow(corr, cmap="RdBu_r", vmin=-cmax, vmax=cmax, aspect="auto")
ax1.set_title(f"{cmb} x {lensing} cross-correlation")
ax1.set_xlabel(f"{lensing} bandpowers")
ax1.set_ylabel(f"{cmb} bandpowers")
fig.colorbar(im1, ax=ax1, fraction=0.046, label="correlation")

fig.tight_layout()
plt.show()

## Appendix: Validating the machinery on small synthetic data

To see the same code path run end-to-end *without* the multi-GB derivative, we feed the low-level
builder a tiny synthetic MFLike-like SACC (one TT spectrum, `lmax = 60`) and a stub lensing
likelihood. This is exactly how the unit tests exercise the glue: extract the windows and fiducial
cosmology from the SACC, run a small CAMB derivative, and contract it with the kernel.

In [ ]:
from types import SimpleNamespace

import numpy as np
import sacc

from soliket.cross_covariance import (
    camb_lensing_derivatives_from_sacc,
    cmb_combs_from_spec_meta,
    cmb_lensing_crosscov,
    lensing_induced_cov,
)


def tiny_mflike_sacc():
    """A minimal MFLike-like SACC: one TT spectrum with bandpower windows + metadata."""
    s = sacc.Sacc()
    s.add_tracer("Misc", "LAT_93_s0", quantity="cmb_temperature", spin=0)
    support = np.arange(2, 30)
    n_bins = 3
    weight = np.zeros((len(support), n_bins))
    centers = []
    for b, idx in enumerate(np.array_split(np.arange(len(support)), n_bins)):
        weight[idx, b] = 1.0 / len(idx)
        centers.append(support[idx].mean())
    s.add_ell_cl(
        "cl_00",
        "LAT_93_s0",
        "LAT_93_s0",
        np.array(centers),
        np.zeros(n_bins),
        window=sacc.BandpowerWindow(support, weight),
    )
    s.metadata["f_sky_LAT"] = 0.4
    s.metadata["cosmo_params"] = repr(
        dict(
            cosmomc_theta=0.0104,
            logA=3.05,
            ombh2=0.0224,
            omch2=0.1202,
            ns=0.9649,
            Alens=1.0,
            tau=0.0544,
        )
    )
    s.metadata["accuracy_params"] = repr({})
    s.metadata["lmax"] = 60
    return s


# A stub lensing likelihood: just the kappa binning matrix and a flat C_ell^phiphi.
lmax_kk = 25
binning = np.zeros((2, lmax_kk))
binning[0, 2:12] = 0.1
binning[1, 12:lmax_kk] = 0.1
lensing_stub = SimpleNamespace(
    binning_matrix=binning,
    provider=SimpleNamespace(
        get_Cl=lambda ell_factor=True: {"pp": np.ones(lmax_kk) * 1e-8}
    ),
)

mflike_sacc = tiny_mflike_sacc()

# The CMB rows: one (ind_camb, support, weight) triple per spectrum, built from
# mflike's own spec_meta so the block rows land in MFLike's data-vector order.
# Here we hand-roll the one-TT-spectrum spec_meta the stub SACC implies; against a
# real likelihood this is just ``cmb_combs_from_spec_meta(mflike.spec_meta)``.
ell, _, ind = mflike_sacc.get_ell_cl(
    "cl_00", "LAT_93_s0", "LAT_93_s0", return_ind=True
)
spec_meta = [
    {
        "pol": "tt",
        "hasYX_xsp": False,
        "t1": "LAT_93",
        "t2": "LAT_93",
        "bpw": mflike_sacc.get_bandpower_windows(ind),
        "leff": ell,
    }
]
combs = cmb_combs_from_spec_meta(spec_meta)

# Each block is built from the same CAMB lensed-Cl derivative; run CAMB once and
# share the bundle across builders via ``derivatives=`` instead of recomputing it
# per block (the derivative is the expensive part, ~3 GB / minutes at full accuracy).
derivs = camb_lensing_derivatives_from_sacc(mflike_sacc)

block = cmb_lensing_crosscov(mflike_sacc, lensing_stub, combs, derivatives=derivs)
print("CMB x lensing block :", block.shape, " finite:", np.all(np.isfinite(block)))

induced = lensing_induced_cov(mflike_sacc, combs, derivatives=derivs)
print(
    "lensing-induced block:",
    induced.shape,
    " symmetric:",
    np.allclose(induced, induced.T),
)

### Wrapping a block in a `CrossCov`

`CrossCov` is the container `MultiGaussianLikelihood` reads. You register each component's
auto-covariance with `add_component`, the off-diagonal block with `add_cross_covariance`, and `save`
it to a SACC file. (`from_cmb_lensing` does all of this for you on real data.)

In [ ]:
import os
import tempfile

from soliket.gaussian.gaussian_data import CrossCov

n_cmb, n_kk = block.shape
xcov = CrossCov()
xcov.add_component("mflike", np.eye(n_cmb))  # placeholder auto-covariances for the demo
xcov.add_component("lensing", np.eye(n_kk))
xcov.add_cross_covariance("mflike", "lensing", block)

path = os.path.join(tempfile.mkdtemp(prefix="soliket_xcov_"), "XCov_demo.fits")
xcov.save(path)

reloaded = CrossCov.load(path)
print("saved to", path)
print("round-trip block matches:", np.allclose(reloaded[("mflike", "lensing")], block))

## 3. The full set of blocks

A complete joint CMB + lensing + LSS analysis needs several blocks, all built from the same CAMB
lensed-Cl derivative (`camb_lensing_derivatives`):

| Block | Builder | What it couples |
| --- | --- | --- |
| CMB x CMB-lensing | `cmb_lensing_crosscov` | MFLike TT/TE/EE x reconstruction kappa-kappa |
| lensing-induced | `lensing_induced_cov` | extra CMB-internal covariance from lensing |
| CMB x shear/galaxy-kappa | `shear_kappa_crosscov` | MFLike x an LSS cross-correlation likelihood |
| N1 reconstruction-noise | `n1_crosscov_block` | the N1 bias contribution to the kappa cross-cov |

The N1 *matrix* itself is produced with the external [`lensitbiases`](https://github.com/carronj/lensitbiases)
package and is **not** part of `soliket.cross_covariance` (only the kernel that *applies* a precomputed
N1 matrix is). See [`../dev/cross_cov/create_cross_covariance.ipynb`](../dev/cross_cov/create_cross_covariance.ipynb)
for the full-accuracy reference build, including N1, and for the committed reference products used in
regression testing.

Next: [`analyse_datasets.ipynb`](analyse_datasets.ipynb) loads a saved `CrossCov` into a
`MultiGaussianLikelihood` and runs a joint analysis.